# Loan default prediction — Bagging Classifier
Run all cells from this folder to train and evaluate a bagging classifier and save `bagging.pkl` beside this notebook. Requires pandas, NumPy, and scikit-learn (tested with scikit-learn 1.7.2).

`Default` is the target (0 = no default, 1 = default). `LoanID` is excluded. Preprocessing is fitted only on the training partition and saved with the classifier. Decision trees are the base learners within the bagging ensemble; no other models are trained.

In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import BaggingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, average_precision_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
DATA_PATH = Path("Loan_data.csv").resolve()
MODEL_PATH = DATA_PATH.with_name("bagging.pkl")
print(f"scikit-learn version: {sklearn.__version__}")

scikit-learn version: 1.7.2


## Load and validate the data

In [2]:
data = pd.read_csv(DATA_PATH)
if "Default" not in data.columns:
    raise ValueError("The dataset must contain the Default target column.")
if data["Default"].isna().any() or set(data["Default"].unique()) != {0, 1}:
    raise ValueError("Default must contain both binary labels 0 and 1 without missing values.")

# Remove identifiers before encoding, avoiding thousands of meaningless features.
data = data.drop(columns=["LoanID"], errors="ignore").drop_duplicates()
X = data.drop(columns="Default")
y = data["Default"].astype(int)
print(f"Rows: {len(data):,}; input features: {X.shape[1]}")
print(f"Missing feature values: {X.isna().sum().sum()}")
print("Target counts:")
print(y.value_counts().sort_index().to_string())

Rows: 255,347; input features: 16
Missing feature values: 0
Target counts:
Default
0    225694
1     29653


## Split before preprocessing
Use a reproducible, stratified 80/20 split to preserve the default rate in both partitions.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
)
numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()
print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")

Training rows: 204,277; test rows: 51,070


## Train the bagging classifier
The ensemble uses 50 bootstrapped decision trees. Limited depth and a minimum leaf size reduce overfitting. Balanced class weights give the minority default class more influence. Numeric scaling is unnecessary for trees. Missing values are imputed, and unseen categories are safely ignored during prediction.

In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_columns),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_columns),
    ],
    remainder="drop",
)

bagging_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", BaggingClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=12,
            min_samples_leaf=20,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        n_estimators=50,
        max_samples=0.8,
        bootstrap=True,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )),
])
bagging_model.fit(X_train, y_train)
print("Bagging classifier training complete.")

Bagging classifier training complete.


## Evaluate on the held-out test set
Accuracy alone can hide missed defaults. Report class-specific precision/recall, balanced accuracy, ROC AUC, and average precision. The test set is reserved for evaluation and is not used to select parameters or thresholds. The saved model is this evaluated training-split model.

In [5]:
train_predictions = bagging_model.predict(X_train)
test_predictions = bagging_model.predict(X_test)
test_probabilities = bagging_model.predict_proba(X_test)[:, 1]
print(f"Training accuracy: {accuracy_score(y_train, train_predictions):.4f}")
print(f"Test accuracy: {accuracy_score(y_test, test_predictions):.4f}")
print(f"Test majority-class baseline accuracy: {y_test.value_counts(normalize=True).max():.4f}")
print(f"Test balanced accuracy: {balanced_accuracy_score(y_test, test_predictions):.4f}")
print(f"Test ROC AUC: {roc_auc_score(y_test, test_probabilities):.4f}")
print(f"Test average precision: {average_precision_score(y_test, test_probabilities):.4f}")
print("\nClassification report:")
print(classification_report(y_test, test_predictions, digits=4, zero_division=0))
print("Confusion matrix (rows = actual, columns = predicted; labels = [0, 1]):")
print(confusion_matrix(y_test, test_predictions, labels=[0, 1]))

Training accuracy: 0.8038
Test accuracy: 0.7710
Test majority-class baseline accuracy: 0.8839
Test balanced accuracy: 0.6769
Test ROC AUC: 0.7466
Test average precision: 0.3133

Classification report:
              precision    recall  f1-score   support

           0     0.9318    0.7995    0.8606     45139
           1     0.2665    0.5544    0.3599      5931

    accuracy                         0.7710     51070
   macro avg     0.5991    0.6769    0.6103     51070
weighted avg     0.8545    0.7710    0.8024     51070

Confusion matrix (rows = actual, columns = predicted; labels = [0, 1]):
[[36088  9051]
 [ 2643  3288]]


## Save and verify `bagging.pkl`
The pickle contains the complete fitted pipeline, including preprocessing. Reload it and pass a pandas DataFrame containing the original input feature columns to `predict` or `predict_proba`.

In [6]:
with MODEL_PATH.open("wb") as model_file:
    pickle.dump(bagging_model, model_file, protocol=pickle.HIGHEST_PROTOCOL)

with MODEL_PATH.open("rb") as model_file:
    loaded_model = pickle.load(model_file)

sample = X_test.head(10)
np.testing.assert_array_equal(loaded_model.predict(sample), test_predictions[:10])
np.testing.assert_allclose(loaded_model.predict_proba(sample)[:, 1], test_probabilities[:10])
print(f"Saved model: {MODEL_PATH}")
print(f"File size: {MODEL_PATH.stat().st_size / (1024 ** 2):.2f} MB")
print("Reload verified: predictions and probabilities match.")

Saved model: /Users/rushipatel/Documents/ML Project/Loan Default Predicition/bagging.pkl
File size: 12.05 MB
Reload verified: predictions and probabilities match.
